# Predicting House Prices with Linear Regression
OIBSIP · Data Analytics · Level 2 · Task 1

**Dataset:** Kaggle *Housing Prices* dataset saved as `Housing.csv` (columns: `price, area, bedrooms, bathrooms, stories, mainroad, guestroom, basement, hotwaterheating, airconditioning, parking, prefarea, furnishingstatus`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

## 1. Load and explore

In [ ]:
df = pd.read_csv("Housing.csv")
print("Shape:", df.shape)
print("\nNull values:\n", df.isnull().sum())
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["price"], kde=True, ax=axes[0]); axes[0].set_title("Price distribution")
sns.histplot(np.log(df["price"]), kde=True, ax=axes[1]); axes[1].set_title("log(price)")
plt.show()

## 2. Feature selection reasoning
Likely strong predictors: **area** (larger homes cost more), **bathrooms**, **bedrooms**, **stories**, **parking**, and **air conditioning**. Location indicators such as **prefarea** and **mainroad** should add a premium. We keep all features and let the model weight them, then inspect coefficients.

## 3. Missing values and encoding

In [ ]:
df = df.dropna()   # dataset has no nulls; this is a safeguard
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
df_enc.head()

## 4. Correlation heatmap

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(df_enc.corr(), annot=True, fmt=".2f", cmap="coolwarm", annot_kws={"size": 7})
plt.show()
print(df_enc.corr()["price"].sort_values(ascending=False))

## 5. Train / test split (80/20) and Linear Regression

In [ ]:
X = df_enc.drop(columns="price")
y = df_enc["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)

mse = mean_squared_error(y_test, pred)
print(f"MSE:  {mse:,.0f}")
print(f"RMSE: {np.sqrt(mse):,.0f}")
print(f"R2:   {r2_score(y_test, pred):.3f}")

## 6. Actual vs predicted

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, alpha=0.6)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual price"); plt.ylabel("Predicted price"); plt.title("Actual vs predicted")
plt.show()

## 7. Residuals

In [ ]:
res = y_test - pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(pred, res, alpha=0.6); axes[0].axhline(0, color="red")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Residual"); axes[0].set_title("Residuals vs predicted")
sns.histplot(res, kde=True, ax=axes[1]); axes[1].set_title("Residual distribution")
plt.show()

Residuals should be centred on zero with no obvious pattern. A funnel shape would suggest the error grows with price (heteroscedasticity).

## 8. Coefficient analysis

In [ ]:
coef = pd.Series(model.coef_, index=X.columns).sort_values()
coef.plot(kind="barh", figsize=(8, 6), title="Linear regression coefficients")
plt.show()
print(coef.sort_values(ascending=False))

## 9. Bonus: Ridge and Lasso

In [ ]:
scaler = StandardScaler().fit(X_train)
Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)
rows = []
for name, m in [("Linear", LinearRegression()), ("Ridge", Ridge(alpha=10)), ("Lasso", Lasso(alpha=10, max_iter=10000))]:
    m.fit(Xtr, y_train)
    p = m.predict(Xte)
    rows.append({"Model": name, "RMSE": np.sqrt(mean_squared_error(y_test, p)), "R2": r2_score(y_test, p)})
pd.DataFrame(rows).set_index("Model").round(3)

## 10. Conclusion
Summarise your R2/RMSE, the strongest price drivers from the coefficients, and limitations (linear assumption, small dataset, no location coordinates).